In [1]:
import json
import os
import random

import h5py
import numpy as np

import time
%matplotlib inline
import matplotlib.pyplot as plt

import robosuite
import imageio
from pprint import pprint   
from lxml import etree

In [2]:
demo_path = "../robosuite/models/assets/demonstrations/1688900181_543158"
hdf5_path = os.path.join(demo_path, "demo.hdf5")
f = h5py.File(hdf5_path, "r")
env_name = f["data"].attrs["env"]
env_info = json.loads(f["data"].attrs["env_info"])
# env_info['camera_names'] = ['frontview', 'birdview', 'agentview', 'robot0_eye_in_hand']
env_info["camera_names"] = "frontview"
env_info["has_renderer"] = True
env_info["use_camera_obs"] = False
env_info["ignore_done"] = True
env_info["has_offscreen_renderer"] = False
env_info["camera_heights"] = 1024
env_info["camera_widths"] = 1024

In [4]:
env = robosuite.make(**env_info)

# list of all demonstrations episodes
demos = list(f["data"].keys())

# print("Playing back random episode... (press ESC to quit)")

# i = 10
# ep = demos[i]
ep = "demo_2"
# os.path.exists(ep) or os.makedirs(ep)
env.reset()

# read the model xml, using the metadata stored in the attribute for this episode
model_xml = f["data/{}".format(ep)].attrs["model_file"]

env.reset()
xml = env.edit_model_xml(model_xml)
env.reset_from_xml_string(xml)
env.sim.reset()

states = f["data/{}/states".format(ep)][()]

# load the actions and play them back open-loop
actions = np.array(f["data/{}/actions".format(ep)][()])

# find obj_to_use
env.sim.set_state_from_flattened(states[-2])
env.step(actions[-1])
obj_id = np.where(env.objects_in_bins==1)[0][0]
obj_to_use = env.objects[obj_id]

env.sim.set_state_from_flattened(states[0])
env.sim.forward()

num_actions = actions.shape[0]
keyframes = []
keyframe_infos = []


In [5]:
for j, action in enumerate(actions):
    obs, reward, done, info = env.step(action)
    env.render()

    obj_geom_id = list(env.obj_geom_id.values())[obj_id][0]
    print(env.sim.data.geom_xpos[obj_geom_id])

    if j < num_actions - 1:
        # ensure that the actions deterministically lead to the same recorded states
        state_playback = env.sim.get_state().flatten()
        state_data = states[j + 1]
        if not np.all(np.equal(state_data, state_playback)):
            err = np.linalg.norm(state_data - state_playback)
            # if err > 2:
            #     print("State mismatch in demo {} at step {}! Error: {}".format(ep, j, err))
            #     break
            # print(f"[warning] playback diverged by {err:.2f} for ep {ep} at step {j}")

[ 0.11122177 -0.21756416  0.86040493]
[ 0.11121268 -0.21757465  0.86040768]
[ 0.11122034 -0.21756338  0.86040963]
[ 0.11123176 -0.21756777  0.86041437]
[ 0.11123936 -0.21757273  0.86040684]
[ 0.11121131 -0.21757966  0.86040739]
[ 0.11120132 -0.21757811  0.86040883]
[ 0.11124176 -0.21758652  0.86040999]
[ 0.1112469  -0.2175935   0.86040966]
[ 0.11122955 -0.2175567   0.86041145]
[ 0.11123081 -0.21755635  0.86041447]
[ 0.11120685 -0.21759053  0.86041197]
[ 0.11124386 -0.21757724  0.86041136]
[ 0.11121438 -0.21759268  0.8604117 ]
[ 0.11124648 -0.21758705  0.86040948]
[ 0.11124976 -0.21755467  0.86041306]
[ 0.11126383 -0.21758944  0.86041523]
[ 0.11125142 -0.21757189  0.86041417]
[ 0.11122178 -0.21755395  0.86041259]
[ 0.11120328 -0.21757024  0.86041192]
[ 0.11119846 -0.21756147  0.86041298]
[ 0.11120968 -0.21755475  0.8604143 ]
[ 0.11121723 -0.21755466  0.86041402]
[ 0.11120251 -0.21756645  0.86041247]
[ 0.11120067 -0.21756412  0.86041309]
[ 0.11121012 -0.2175908   0.86041184]
[ 0.11124258

In [15]:
len(actions)

722

In [16]:
env.sim.set_state_from_flattened(states[-2])
env.step(actions[-1])
obj_id = np.where(env.objects_in_bins==1)[0][0]
obj_to_use = env.objects[obj_id]

env.sim.set_state_from_flattened(states[350])
env.sim.forward()
obj_geom_id = list(env.obj_geom_id.values())[obj_id][0]

obj_pos = env.sim.data.geom_xpos[obj_geom_id]
for geoms in env.obj_geom_id.values():
    if geoms[0] == obj_geom_id:
        continue
    pos = env.sim.data.geom_xpos[geoms[0]]
    dist = np.linalg.norm(pos - obj_pos)
    print(dist)

0.12423491231881739
0.12135332733080982
0.12679644009997634
0.0538627958894629
0.14519654457877768


In [17]:
env.bin1_pos, env.bin2_pos, obj_pos

(array([ 0.1 , -0.25,  0.8 ]),
 array([0.1 , 0.28, 0.8 ]),
 array([ 0.11231349, -0.21983659,  0.86010401]))

In [ ]:
env.sim.model.geom_pos.shape, len(list(env.sim.model._geom_id2name.keys()))

((122, 3), 122)

array([ 5.89018274e-05, -1.33653727e-04,  3.58399591e-04])

In [ ]:
list(env.object_to_id.keys())

['milk', 'bread', 'cereal', 'can', 'lemon', 'bottle']